In [ ]:
import sympy as sp

# ============================================================
# Pretty symbols for Stevens parameters
# ============================================================

B20, B22, B40, B42, B43, B44 = sp.symbols(
    "B20 B22 B40 B42 B43 B44", real=True
)

# Optional rank-6 symbols if needed later
B60, B62, B63, B64, B66 = sp.symbols(
    "B60 B62 B63 B64 B66", real=True
)


# ============================================================
# Helpers
# ============================================================

def ket_label(m):
    """
    Pretty ket label for m_J.
    """
    m = sp.Rational(m)
    if m.q == 1:
        return rf"\ket{{{m.p}}}"
    return rf"\ket{{\frac{{{m.p}}}{{{m.q}}}}}"


def make_basis(J, ordering="descending"):
    """
    Build a spin basis.

    ordering = "descending":
        |J>, |J-1>, ..., |-J>

    ordering = list:
        custom basis, e.g.
        [5/2, 1/2, -3/2, -5/2, -1/2, 3/2]
    """
    J = sp.Rational(J)

    if isinstance(ordering, (list, tuple)):
        return [sp.Rational(x) for x in ordering]

    n = int(2 * J + 1)

    if ordering == "descending":
        return [J - i for i in range(n)]

    if ordering == "ascending":
        return [-J + i for i in range(n)]

    raise ValueError("ordering must be 'descending', 'ascending', or a custom list.")


def angular_momentum_matrices(J, ordering="descending"):
    """
    Construct symbolic Jz, Jplus, Jminus matrices.
    Matrix convention:
        M[row, col] = <m_row| operator |m_col>
    """
    J = sp.Rational(J)
    basis = make_basis(J, ordering)
    n = len(basis)

    Jz = sp.zeros(n)
    Jp = sp.zeros(n)
    Jm = sp.zeros(n)

    m_to_idx = {m: i for i, m in enumerate(basis)}

    for i, m in enumerate(basis):
        Jz[i, i] = m

    for col, m in enumerate(basis):
        mp = m + 1
        mm = m - 1

        if mp in m_to_idx:
            row = m_to_idx[mp]
            Jp[row, col] = sp.sqrt(J * (J + 1) - m * (m + 1))

        if mm in m_to_idx:
            row = m_to_idx[mm]
            Jm[row, col] = sp.sqrt(J * (J + 1) - m * (m - 1))

    return basis, Jz, Jp, Jm


# ============================================================
# Stevens operators
# ============================================================

def stevens_operators_symbolic(J, ordering="descending"):
    """
    Construct analytical Stevens operator matrices.

    Includes:
        O20, O22,
        O40, O42, O43, O44,
        O60, O62, O63, O64, O66
    """
    basis, Jz, Jp, Jm = angular_momentum_matrices(J, ordering)

    n = len(basis)
    I = sp.eye(n)
    J = sp.Rational(J)
    X = J * (J + 1)

    Jz2 = Jz**2
    Jz3 = Jz**3
    Jz4 = Jz**4
    Jz6 = Jz**6

    Jp2 = Jp**2
    Jm2 = Jm**2
    Jp3 = Jp**3
    Jm3 = Jm**3
    Jp4 = Jp**4
    Jm4 = Jm**4
    Jp6 = Jp**6
    Jm6 = Jm**6

    # ----------------------------
    # Rank 2
    # ----------------------------
    O20 = 3 * Jz2 - X * I
    O22 = sp.Rational(1, 2) * (Jp2 + Jm2)

    # ----------------------------
    # Rank 4
    # ----------------------------
    O40 = (
        35 * Jz4
        - (30 * X - 25) * Jz2
        + (3 * X**2 - 6 * X) * I
    )

    O42 = sp.Rational(1, 4) * (
        (7 * Jz2 - X * I - 5 * I) * (Jp2 + Jm2)
        + (Jp2 + Jm2) * (7 * Jz2 - X * I - 5 * I)
    )

    O43 = sp.Rational(1, 4) * (
        Jz * (Jp3 + Jm3)
        + (Jp3 + Jm3) * Jz
    )

    O44 = sp.Rational(1, 2) * (Jp4 + Jm4)

    # ----------------------------
    # Rank 6
    # Included for completeness.
    # For J = 5/2 these vanish or are dependent in the CEF sense.
    # ----------------------------
    O60 = (
        231 * Jz6
        - (315 * X - 735) * Jz4
        + (105 * X**2 - 525 * X + 294) * Jz2
        + (-5 * X**3 + 40 * X**2 - 60 * X) * I
    )

    O62 = sp.Rational(1, 4) * (
        (
            33 * Jz4
            - (18 * X + 123) * Jz2
            + (X**2 + 10 * X + 102) * I
        ) * (Jp2 + Jm2)
        + (Jp2 + Jm2) * (
            33 * Jz4
            - (18 * X + 123) * Jz2
            + (X**2 + 10 * X + 102) * I
        )
    )

    O63 = sp.Rational(1, 4) * (
        (11 * Jz3 - (3 * X + 59) * Jz) * (Jp3 + Jm3)
        + (Jp3 + Jm3) * (11 * Jz3 - (3 * X + 59) * Jz)
    )

    O64 = sp.Rational(1, 4) * (
        (11 * Jz2 - X * I - 38 * I) * (Jp4 + Jm4)
        + (Jp4 + Jm4) * (11 * Jz2 - X * I - 38 * I)
    )

    O66 = sp.Rational(1, 2) * (Jp6 + Jm6)

    ops = {
        "O20": sp.simplify(O20),
        "O22": sp.simplify(O22),
        "O40": sp.simplify(O40),
        "O42": sp.simplify(O42),
        "O43": sp.simplify(O43),
        "O44": sp.simplify(O44),
        "O60": sp.simplify(O60),
        "O62": sp.simplify(O62),
        "O63": sp.simplify(O63),
        "O64": sp.simplify(O64),
        "O66": sp.simplify(O66),
    }

    return basis, ops


# ============================================================
# Symmetry Hamiltonians
# ============================================================

def analytical_H(J, symmetry, ordering="descending", convention="standard"):
    """
    Build analytical CEF Hamiltonian for a given symmetry.

    symmetry:
        "D2", "D3", "D4"

    convention:
        "standard":
            H = B20 O20 + B40 O40 + ...
            This gives for J=5/2:
                O40(|±5/2>) = 60
                O40(|±1/2>) = 120
                O40(|±3/2>) = -180

        "flipped_B40":
            Uses -B40 O40 instead of +B40 O40.
            This gives your D4 convention:
                A = 10B20 - 60B40
                D = -8B20 - 120B40
                F = -2B20 + 180B40
    """
    basis, ops = stevens_operators_symbolic(J, ordering)

    if convention == "standard":
        s40 = 1
    elif convention == "flipped_B40":
        s40 = -1
    else:
        raise ValueError("convention must be 'standard' or 'flipped_B40'.")

    if symmetry == "D2":
        H = (
            B20 * ops["O20"]
            + B22 * ops["O22"]
            + s40 * B40 * ops["O40"]
            + B42 * ops["O42"]
            + B44 * ops["O44"]
        )

    elif symmetry == "D3":
        H = (
            B20 * ops["O20"]
            + s40 * B40 * ops["O40"]
            + B43 * ops["O43"]
        )

    elif symmetry == "D4":
        H = (
            B20 * ops["O20"]
            + s40 * B40 * ops["O40"]
            + B44 * ops["O44"]
        )

    else:
        raise ValueError("symmetry must be 'D2', 'D3', or 'D4'.")

    H = sp.simplify(H)
    return H, basis, ops


def print_basis(basis):
    print("Basis:")
    for i, m in enumerate(basis):
        print(f"  {i}: |{sp.latex(m)}>")


def latex_matrix(H):
    """
    Return LaTeX bmatrix for the symbolic Hamiltonian.
    """
    return sp.latex(H, mat_delim="[", mat_str="matrix")


# ============================================================
# Custom bases matching your analytical blocks
# ============================================================

basis_D2_block = [
    sp.Rational(5, 2),
    sp.Rational(1, 2),
    sp.Rational(-3, 2),
    sp.Rational(-5, 2),
    sp.Rational(-1, 2),
    sp.Rational(3, 2),
]

basis_D4_block = [
    sp.Rational(5, 2),
    sp.Rational(1, 2),
    sp.Rational(-3, 2),
    sp.Rational(-5, 2),
    sp.Rational(-1, 2),
    sp.Rational(3, 2),
]

basis_D3_code = [
    sp.Rational(5, 2),
    sp.Rational(1, 2),
    sp.Rational(-1, 2),
    sp.Rational(3, 2),
    sp.Rational(-5, 2),
    sp.Rational(-3, 2),
]





D2 basis
Basis:
  0: |\frac{5}{2}>
  1: |\frac{1}{2}>
  2: |- \frac{3}{2}>
  3: |- \frac{5}{2}>
  4: |- \frac{1}{2}>
  5: |\frac{3}{2}>

D2 analytical Hamiltonian:
⎡ 10⋅B₂₀ - 60⋅B₄₀   √10⋅(B₂₂ + 9⋅B₄₂)       12⋅√5⋅B₄₄               0          ↪
⎢                                                                              ↪
⎢√10⋅(B₂₂ + 9⋅B₄₂)   -8⋅B₂₀ - 120⋅B₄₀   3⋅√2⋅(B₂₂ - 5⋅B₄₂)          0          ↪
⎢                                                                              ↪
⎢    12⋅√5⋅B₄₄      3⋅√2⋅(B₂₂ - 5⋅B₄₂)   -2⋅B₂₀ + 180⋅B₄₀           0          ↪
⎢                                                                              ↪
⎢        0                  0                   0            10⋅B₂₀ - 60⋅B₄₀   ↪
⎢                                                                              ↪
⎢        0                  0                   0           √10⋅(B₂₂ + 9⋅B₄₂)  ↪
⎢                                                                              ↪
⎣        0               

In [3]:
# ============================================================
# Generate analytical matrices
# ============================================================

# Use convention="flipped_B40" if you want your D4 signs:
# A = 10B20 - 60B40, etc.
CONVENTION = "standard"

H_D2, basis_D2, ops_D2 = analytical_H(
    J=sp.Rational(5, 2),
    symmetry="D2",
    ordering=basis_D2_block,
    convention=CONVENTION,
)

H_D3, basis_D3, ops_D3 = analytical_H(
    J=sp.Rational(5, 2),
    symmetry="D3",
    ordering=basis_D3_code,
    convention=CONVENTION,
)

H_D4, basis_D4, ops_D4 = analytical_H(
    J=sp.Rational(5, 2),
    symmetry="D4",
    ordering=basis_D4_block,
    convention=CONVENTION,
)

print("\nD2 basis")
print_basis(basis_D2)
print("\nD2 analytical Hamiltonian:")
sp.pretty_print(H_D2)

print("\nD3 basis")
print_basis(basis_D3)
print("\nD3 analytical Hamiltonian:")
sp.pretty_print(H_D3)

print("\nD4 basis")
print_basis(basis_D4)
print("\nD4 analytical Hamiltonian:")
sp.pretty_print(H_D4)

print("\nLaTeX D2:")
print(latex_matrix(H_D2))

print("\nLaTeX D3:")
print(latex_matrix(H_D3))

print("\nLaTeX D4:")
print(latex_matrix(H_D4))


D2 basis
Basis:
  0: |\frac{5}{2}>
  1: |\frac{1}{2}>
  2: |- \frac{3}{2}>
  3: |- \frac{5}{2}>
  4: |- \frac{1}{2}>
  5: |\frac{3}{2}>

D2 analytical Hamiltonian:
⎡ 10⋅B₂₀ + 60⋅B₄₀   √10⋅(B₂₂ + 9⋅B₄₂)       12⋅√5⋅B₄₄               0          ↪
⎢                                                                              ↪
⎢√10⋅(B₂₂ + 9⋅B₄₂)   -8⋅B₂₀ + 120⋅B₄₀   3⋅√2⋅(B₂₂ - 5⋅B₄₂)          0          ↪
⎢                                                                              ↪
⎢    12⋅√5⋅B₄₄      3⋅√2⋅(B₂₂ - 5⋅B₄₂)   -2⋅B₂₀ - 180⋅B₄₀           0          ↪
⎢                                                                              ↪
⎢        0                  0                   0            10⋅B₂₀ + 60⋅B₄₀   ↪
⎢                                                                              ↪
⎢        0                  0                   0           √10⋅(B₂₂ + 9⋅B₄₂)  ↪
⎢                                                                              ↪
⎣        0               

In [1]:
import sympy as sp

# ============================================================
# Stevens parameter symbols
# ============================================================

B20, B22, B40, B42, B43, B44 = sp.symbols(
    "B20 B22 B40 B42 B43 B44", real=True
)

PARAM_TO_OPERATOR = {
    "B20": "O20",
    "B22": "O22",
    "B40": "O40",
    "B42": "O42",
    "B43": "O43",
    "B44": "O44",
}

SYMMETRY_ALLOWED = {
    "D2": ["B20", "B22", "B40", "B42", "B44"],
    "D3": ["B20", "B40", "B43"],
    "D4": ["B20", "B40", "B44"],
}

PARAM_SYMBOL = {
    "B20": B20,
    "B22": B22,
    "B40": B40,
    "B42": B42,
    "B43": B43,
    "B44": B44,
}


# ============================================================
# Basis helpers
# ============================================================

def R(x):
    return sp.Rational(x)

def ket_label(m):
    m = sp.Rational(m)
    if m.q == 1:
        return rf"\ket{{{m.p}}}"
    return rf"\ket{{\frac{{{m.p}}}{{{m.q}}}}}"

def print_basis(basis):
    print("Basis used:")
    for i, m in enumerate(basis):
        print(f"  index {i}: {ket_label(m)}")

def latex_matrix(M):
    return sp.latex(M, mat_delim="[", mat_str="matrix")


# ============================================================
# Hutchings table entries for J = 5/2
# Actual matrix elements, after applying the table F factors
# ============================================================

J52 = sp.Rational(5, 2)

TABLE_ENTRIES = {
    J52: {
        # ----------------------------------------------------
        # Diagonal operators
        # From Hutchings O_2^0 table:
        # F = 2, entries: ±1/2=-4, ±3/2=-1, ±5/2=5
        # actual: -8, -2, 10
        # ----------------------------------------------------
        "O20_diag": {
            sp.Rational(1, 2): -8,
            sp.Rational(-1, 2): -8,
            sp.Rational(3, 2): -2,
            sp.Rational(-3, 2): -2,
            sp.Rational(5, 2): 10,
            sp.Rational(-5, 2): 10,
        },

        # ----------------------------------------------------
        # From Hutchings O_4^0 table:
        # F = 60, entries: ±1/2=2, ±3/2=-3, ±5/2=1
        # actual: 120, -180, 60
        # ----------------------------------------------------
        "O40_diag": {
            sp.Rational(1, 2): 120,
            sp.Rational(-1, 2): 120,
            sp.Rational(3, 2): -180,
            sp.Rational(-3, 2): -180,
            sp.Rational(5, 2): 60,
            sp.Rational(-5, 2): 60,
        },

        # ----------------------------------------------------
        # Off-diagonal operators
        # Stored as unordered Hermitian pairs:
        # (m1, m2): <m1|O|m2>
        # The code fills both H[m1,m2] and H[m2,m1].
        # ----------------------------------------------------

        # O_2^2 = 1/2 (J_+^2 + J_-^2)
        "O22_offdiag": {
            # <±3/2 || ∓1/2> = 3 sqrt(2)
            (sp.Rational(3, 2), sp.Rational(-1, 2)): 3 * sp.sqrt(2),
            (sp.Rational(-3, 2), sp.Rational(1, 2)): 3 * sp.sqrt(2),

            # <±5/2 || ±1/2> = sqrt(10)
            (sp.Rational(5, 2), sp.Rational(1, 2)): sp.sqrt(10),
            (sp.Rational(-5, 2), sp.Rational(-1, 2)): sp.sqrt(10),
        },

        # O_4^2 table has F = 3 for J=5/2
        # listed entries:
        # <±3/2 || ∓1/2> = -5 sqrt(2)
        # <±5/2 || ±1/2> = 3 sqrt(10)
        # actual after F=3:
        # -15 sqrt(2), 9 sqrt(10)
        "O42_offdiag": {
            (sp.Rational(3, 2), sp.Rational(-1, 2)): -15 * sp.sqrt(2),
            (sp.Rational(-3, 2), sp.Rational(1, 2)): -15 * sp.sqrt(2),

            (sp.Rational(5, 2), sp.Rational(1, 2)): 9 * sp.sqrt(10),
            (sp.Rational(-5, 2), sp.Rational(-1, 2)): 9 * sp.sqrt(10),
        },

        # O_4^3 table has F = 3 for J=5/2
        # listed entry <5/2 || -1/2> = sqrt(10)
        # actual: 3 sqrt(10)
        # The partner has the opposite sign from the O_4^3 operator.
        "O43_offdiag": {
            (sp.Rational(5, 2), sp.Rational(-1, 2)): 3 * sp.sqrt(10),
            (sp.Rational(1, 2), sp.Rational(-5, 2)): -3 * sp.sqrt(10),
        },

        # O_4^4 table has F = 12 for J=5/2
        # listed entry <5/2 || -3/2> = sqrt(5)
        # actual: 12 sqrt(5)
        "O44_offdiag": {
            (sp.Rational(5, 2), sp.Rational(-3, 2)): 12 * sp.sqrt(5),
            (sp.Rational(-5, 2), sp.Rational(3, 2)): 12 * sp.sqrt(5),
        },
    }
}


# ============================================================
# Build operator matrix directly from encoded table entries
# ============================================================

def operator_from_hutchings_table(J, op_name, basis):
    J = sp.Rational(J)

    if J not in TABLE_ENTRIES:
        raise ValueError(
            f"No table entries encoded for J={J}. "
            "Add the Hutchings row for this J to TABLE_ENTRIES."
        )

    data = TABLE_ENTRIES[J]
    n = len(basis)
    basis = [sp.Rational(m) for m in basis]
    m_to_i = {m: i for i, m in enumerate(basis)}

    O = sp.zeros(n)

    if op_name == "O20":
        for m, val in data["O20_diag"].items():
            if m in m_to_i:
                O[m_to_i[m], m_to_i[m]] = val

    elif op_name == "O40":
        for m, val in data["O40_diag"].items():
            if m in m_to_i:
                O[m_to_i[m], m_to_i[m]] = val

    else:
        key = f"{op_name}_offdiag"
        if key not in data:
            raise ValueError(f"{op_name} not encoded for J={J}.")

        for (m1, m2), val in data[key].items():
            if m1 in m_to_i and m2 in m_to_i:
                i = m_to_i[m1]
                j = m_to_i[m2]
                O[i, j] = val
                O[j, i] = val

    return sp.simplify(O)


def analytical_H_from_tables(J, symmetry, basis):
    """
    Build H_CEF directly from the encoded Hutchings table entries.

    H = sum B_l^m O_l^m

    No sign flipping.
    No convention adjustment.
    This is table-based.
    """
    if symmetry not in SYMMETRY_ALLOWED:
        raise ValueError(f"Unknown symmetry: {symmetry}")

    basis = [sp.Rational(m) for m in basis]
    H = sp.zeros(len(basis))

    for param in SYMMETRY_ALLOWED[symmetry]:
        op_name = PARAM_TO_OPERATOR[param]
        Bsym = PARAM_SYMBOL[param]
        O = operator_from_hutchings_table(J, op_name, basis)
        H += Bsym * O

    return sp.simplify(H)


# ============================================================
# Useful bases
# ============================================================

basis_D2_block = [
    sp.Rational(5, 2),
    sp.Rational(1, 2),
    sp.Rational(-3, 2),
    sp.Rational(-5, 2),
    sp.Rational(-1, 2),
    sp.Rational(3, 2),
]

basis_D4_block = [
    sp.Rational(5, 2),
    sp.Rational(1, 2),
    sp.Rational(-3, 2),
    sp.Rational(-5, 2),
    sp.Rational(-1, 2),
    sp.Rational(3, 2),
]

basis_D3_block = [
    sp.Rational(5, 2),
    sp.Rational(-1, 2),
    sp.Rational(1, 2),
    sp.Rational(-5, 2),
    sp.Rational(3, 2),
    sp.Rational(-3, 2),
]



In [2]:
#============================================================
# Generate analytical matrices from tables
# ============================================================

H_D2_table = analytical_H_from_tables(
    J=sp.Rational(5, 2),
    symmetry="D2",
    basis=basis_D2_block,
)

H_D3_table = analytical_H_from_tables(
    J=sp.Rational(5, 2),
    symmetry="D3",
    basis=basis_D3_block,
)

H_D4_table = analytical_H_from_tables(
    J=sp.Rational(5, 2),
    symmetry="D4",
    basis=basis_D4_block,
)


print("\nD2 basis:")
print_basis(basis_D2_block)
print("\nD2 Hamiltonian from Hutchings table:")
sp.pretty_print(H_D2_table)
print("\nLaTeX D2:")
print(latex_matrix(H_D2_table))


print("\nD3 basis:")
print_basis(basis_D3_block)
print("\nD3 Hamiltonian from Hutchings table:")
sp.pretty_print(H_D3_table)
print("\nLaTeX D3:")
print(latex_matrix(H_D3_table))


print("\nD4 basis:")
print_basis(basis_D4_block)
print("\nD4 Hamiltonian from Hutchings table:")
sp.pretty_print(H_D4_table)
print("\nLaTeX D4:")
print(latex_matrix(H_D4_table))


D2 basis:
Basis used:
  index 0: \ket{\frac{5}{2}}
  index 1: \ket{\frac{1}{2}}
  index 2: \ket{\frac{-3}{2}}
  index 3: \ket{\frac{-5}{2}}
  index 4: \ket{\frac{-1}{2}}
  index 5: \ket{\frac{3}{2}}

D2 Hamiltonian from Hutchings table:
⎡ 10⋅B₂₀ + 60⋅B₄₀   √10⋅(B₂₂ + 9⋅B₄₂)       12⋅√5⋅B₄₄               0          ↪
⎢                                                                              ↪
⎢√10⋅(B₂₂ + 9⋅B₄₂)   -8⋅B₂₀ + 120⋅B₄₀   3⋅√2⋅(B₂₂ - 5⋅B₄₂)          0          ↪
⎢                                                                              ↪
⎢    12⋅√5⋅B₄₄      3⋅√2⋅(B₂₂ - 5⋅B₄₂)   -2⋅B₂₀ - 180⋅B₄₀           0          ↪
⎢                                                                              ↪
⎢        0                  0                   0            10⋅B₂₀ + 60⋅B₄₀   ↪
⎢                                                                              ↪
⎢        0                  0                   0           √10⋅(B₂₂ + 9⋅B₄₂)  ↪
⎢                                